***PDF READER AI CHATBOT***

Using the provided RAG Pipeline Guidline.

Importing relevant packages

In [ ]:
#Install the following packages to allow the notebook to work

# pip install langchain langchain-community langchain-huggingface faiss-cpu pypdf sentence-transformers


# Imports

import os
import glob
import time
import numpy as np
import pandas as pd
from pypdf.errors import PdfReadError, PdfStreamError
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


C:\Users\madha\AppData\Local\Temp\ipykernel_9460\1209610469.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Document Loading, Chunking and Stroing to FAISS database phase.

In [16]:

# Loading PDF
PDF_FOLDER_PATH = "./PDF Test Files"  # Put all your PDFs inside this folder
FAISS_INDEX_PATH = "./faiss_naive_index"

# Initialize a standard, lightweight embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Grab the file paths of all PDFs currently sitting in PDF test folder path
current_pdfs = glob.glob(os.path.join(PDF_FOLDER_PATH, "*.pdf"))

def load_pdf_documents(pdf_path):
    try:
        return PyPDFLoader(pdf_path).load()
    except (PdfReadError, PdfStreamError, OSError) as error:
        print(
            f"ERROR: Could not read PDF '{pdf_path}'. It may be invalid, damaged, or inaccessible. "
            f"Details: {error}"
        )
        return []

if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS database...")
    vector_store = FAISS.load_local(
        FAISS_INDEX_PATH, 
        embeddings, 
        allow_dangerous_deserialization=True
    )
    
    # Look inside FAISS to see which PDF file paths have ALREADY been embedded
    existing_pdfs = {
        doc.metadata.get("source") 
        for doc in vector_store.docstore._dict.values()
    }
    
    #  Compare current folder PDFs against existing FAISS PDFs
    new_pdfs = [pdf for pdf in current_pdfs if pdf not in existing_pdfs]
    
    # If new PDFs are found, embed ONLY the new ones and append them!
    if new_pdfs:
        print(f"Found {len(new_pdfs)} new PDF(s). Adding to existing database...")
        new_docs = []
        for pdf_path in new_pdfs:
            print(f"Loading: {pdf_path}")
            new_docs.extend(load_pdf_documents(pdf_path))
        
        if new_docs:
            new_chunks = text_splitter.split_documents(new_docs)
            vector_store.add_documents(new_chunks) # Appends without rebuilding!
            vector_store.save_local(FAISS_INDEX_PATH) # Overwrites save file with updated DB
            print("Database successfully updated!")
        else:
            print("ERROR: No readable pages were found in the new PDF files. The database was not changed.")
    else:
        print("No new PDFs detected. Database is ready!")

else:
    # Runs ONLY the very first time when no database exists yet
    print("Building FAISS database for the first time...")
    t0 = time.perf_counter() # PDF text extraction timer
    all_docs = []
    for pdf_path in current_pdfs:
        print(f"Loading: {pdf_path}")
        all_docs.extend(load_pdf_documents(pdf_path))
    pdf_load_time = time.perf_counter() - t0 #PDF load time
    
    if not all_docs:
        raise RuntimeError("No readable PDF pages were found. Check that the PDF files are valid and not damaged.")
        
    t1 = time.perf_counter() #Text chunking timer start
    chunks = text_splitter.split_documents(all_docs)
    chunk_time = time.perf_counter() - t1 #chunking timer finish
    
    t2 = time.perf_counter()# embedding timer start
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(FAISS_INDEX_PATH)
    embed_index_time = time.perf_counter() - t2
    
    # Save ingestion timings to a CSV for your presentation
    pd.DataFrame([{
        "Pipeline_Version": "Naive RAG (v1)",
        "Total_PDFs": len(current_pdfs),
        "Total_Pages": len(all_docs),
        "Total_Chunks": len(chunks),
        "PDF_Extraction_Time_s": round(pdf_load_time, 3),
        "Chunking_Time_s": round(chunk_time, 3),
        "Embedding_Indexing_Time_s": round(embed_index_time, 3),
        "Total_Ingestion_Time_s": round(pdf_load_time + chunk_time + embed_index_time, 3)
    }]).to_csv("naive_rag_ingestion_timing.csv", index=False)
    
    
    print("Initial database built and saved!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4815.50it/s]


Building FAISS database for the first time...
Loading: ./PDF Test Files\Attention_Is_All_You_Need_Transformer.pdf
Loading: ./PDF Test Files\Llama_3_Herd_of_Models_Report_92_Pages.pdf
Loading: ./PDF Test Files\LLM_Survey_140_Pages.pdf
Loading: ./PDF Test Files\Modular_RAG_Comprehensive_Survey_2024.pdf
Loading: ./PDF Test Files\NIST_AI_Risk_Management_Framework_Policy.pdf
Loading: ./PDF Test Files\Original_RAG_Paper_Lewis_2020.pdf
Initial database built and saved!


Finding relevant context from user prompt to send to LLM


In [17]:

# Initialize local LLM 
llm = Ollama(model="llama3")

# Retrieves top 4 chunks
retriever = vector_store.as_retriever(search_kwargs = {"k" : 4})

# Define a rigid, static prompt template
prompt_template = """
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 

Context: {context}

Question: {question}

Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)

# Helper function to format the retrieved documents as a single string
def format_docs_with_sources(docs):
    formatted_chunks = []
    for doc in docs:
        source_file = os.path.basename(doc.metadata.get("source", "Unknown PDF"))
        page_num = doc.metadata.get("page", 0) + 1
        formatted_chunks.append(
            f"[Source: {source_file}, Page: {page_num}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(formatted_chunks)

# Build the LangChain Expression Language (LCEL) chain
naive_rag_chain = (
    {"context": retriever | format_docs_with_sources, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Ask Questions

In [18]:
# Invoke the chain
question = "Give me the key points of the Nation address document."
response = naive_rag_chain.invoke(question)

print(response)

I don't have the key points of the Nation address document, as the context provided does not appear to contain information about a national address document. The provided context seems to be related to AI risk management frameworks, policies, and documents. If you meant to ask about the NIST AI Risk Management Framework Policy document, I can try to summarize the key points from the provided pages:

* The document emphasizes the importance of having a purpose-driven culture focused on risk understanding and management.
* The GOVERN function is described as a critical component of the AI risk management framework, which involves having policies, processes, procedures, and practices in place to manage AI risks.
* The GOVERN function also provides opportunities for organizations to clarify and define roles and responsibilities for humans in the Human-AI team configurations and to create mechanisms for explicit decision-making processes.
* The document suggests that implementing the GOVERN

***Testing***

Everything beyond this point is testing to measure how well this RAG implementation is.

Now we will test this RAG Pipeline with the following test metrics:
1. Retrieval Latency (sec): How fast FAISS finds and pulls the 4 text chunks.   
2. Generation Latency (sec): How long llama3 takes to read the chunks and generate the answer.
3. Context Relevance (0.0–1.0): Are the retrieved PDF chunks mathematically close in meaning to the question?   C
4. Context Keyword Recall (0.0–1.0): Did the retriever actually pull the specific facts/names needed to answer the question?
5. Answer Similarity to Ground Truth (0.0–1.0): How closely does llama3's final answer match your ideal written answer?
6. Faithfulness / Hallucination Score (0.0–1.0): Did llama3 stick strictly to the PDF context, or did it make up unsupported details?



In [19]:
# Define the Test Dataset Dictionary (Questions, Ideal Answers, and Must-Have Keywords)


test_dataset = [
    # 1. Original_RAG_Paper_Lewis_2020.pdf
    {
        "category": "Specific Architecture (Original RAG Paper)",
        "question": "In the original Retrieval-Augmented Generation (RAG) paper, what are the two RAG model formulations proposed and how do they differ?",
        "ground_truth": "The two formulations are RAG-Sequence and RAG-Token. RAG-Sequence uses the same retrieved document to generate the complete target sequence, whereas RAG-Token can draw from different retrieved passages for each target token.",
        "expected_keywords": ["RAG-Sequence", "RAG-Token", "parametric", "non-parametric"]
    },
    # 2. Original_RAG_Paper_Lewis_2020.pdf
    {
        "category": "Exact Technical Spec (Original RAG Paper)",
        "question": "What pre-trained models are used for the parametric memory and the non-parametric retriever in the original RAG paper?",
        "ground_truth": "The parametric memory uses a pre-trained BART-large sequence-to-sequence model, and the non-parametric memory uses a Dense Passage Retriever (DPR) over a Wikipedia dense vector index.",
        "expected_keywords": ["BART", "Dense Passage Retriever", "DPR", "Wikipedia"]
    },
    # 3. Modular_RAG_Comprehensive_Survey_2024.pdf
    {
        "category": "Broad Summary (Modular RAG Survey)",
        "question": "According to the RAG survey paper, what are the three main developmental paradigms of Retrieval-Augmented Generation and how do they evolve?",
        "ground_truth": "The three paradigms are Naive RAG, Advanced RAG, and Modular RAG. Naive RAG follows a basic indexing, retrieval, and generation flow; Advanced RAG adds pre-retrieval and post-retrieval optimizations like reranking; Modular RAG introduces flexible modules, routing, and iterative workflows.",
        "expected_keywords": ["Naive RAG", "Advanced RAG", "Modular RAG", "rerank"]
    },
    # 4. Llama_3_Herd_of_Models_Report_92_Pages.pdf
    {
        "category": "Exact Numbers & Specs (Llama 3 Report)",
        "question": "What are the parameter sizes of the three multilingual Llama 3 foundation models, and how many tokens was the largest model pre-trained on?",
        "ground_truth": "The three Llama 3 foundation models have 8B, 70B, and 405B parameters. The largest 405B parameter model was pre-trained on 15.6 trillion tokens with a context window of up to 128K tokens.",
        "expected_keywords": ["8B", "70B", "405B", "15.6", "128K"]
    },
    # 5. Attention_Is_All_You_Need_Transformer.pdf
    {
        "category": "Architecture Detail (Transformer Paper)",
        "question": "In the 'Attention Is All You Need' paper, how many identical layers are in the Transformer encoder and decoder stacks, and how many parallel attention heads are used?",
        "ground_truth": "Both the encoder and decoder stacks are composed of N = 6 identical layers, and the base Transformer model employs h = 8 parallel attention layers, or heads, with d_model = 512.",
        "expected_keywords": ["6", "8", "512", "Multi-Head Attention", "Scaled Dot-Product"]
    },
    # 6. LLM_Survey_140_Pages.pdf
    {
        "category": "List Extraction (LLM Survey 140 Pages)",
        "question": "What are the three representative emergent abilities of Large Language Models discussed in the Survey of Large Language Models?",
        "ground_truth": "The three typical emergent abilities of Large Language Models are in-context learning (ICL), instruction following, and step-by-step reasoning (such as chain-of-thought prompting).",
        "expected_keywords": ["in-context learning", "instruction following", "step-by-step reasoning"]
    },
    # 7. NIST_AI_Risk_Management_Framework_Policy.pdf
    {
        "category": "Policy Guideline Lookup (NIST AI RMF)",
        "question": "What are the four high-level core functions that make up the NIST AI Risk Management Framework (AI RMF) Core?",
        "ground_truth": "The NIST AI RMF Core is composed of four core functions: GOVERN, MAP, MEASURE, and MANAGE.",
        "expected_keywords": ["GOVERN", "MAP", "MEASURE", "MANAGE"]
    },
    # 8. NIST_AI_Risk_Management_Framework_Policy.pdf (Trustworthiness Characteristics)
    {
        "category": "Multi-Item Policy List (NIST AI RMF)",
        "question": "According to the NIST AI Risk Management Framework, what are the key characteristics of trustworthy AI systems?",
        "ground_truth": "Characteristics of trustworthy AI systems include being valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed.",
        "expected_keywords": ["valid and reliable", "safe", "accountable", "explainable", "fair"]
    },
    # 9. Multi-Document Synthesis (Transformer Paper + Llama 3 Report)
    {
        "category": "Multi-Document Synthesis",
        "question": "How does the Transformer attention mechanism described in 'Attention Is All You Need' relate to the attention architecture used in the Llama 3 models?",
        "ground_truth": "The Transformer paper introduces standard Multi-Head Attention in an encoder-decoder architecture, while Llama 3 uses a dense decoder-only Transformer architecture that adopts Grouped-Query Attention (GQA) with 8 key-value heads to improve inference speed and reduce key-value cache size.",
        "expected_keywords": ["Transformer", "Multi-Head", "Grouped-Query Attention", "decoder"]
    },
    # 10. Out-of-Scope Trick Question (Tests Hallucination / Abstention)
    {
        "category": "Out-of-Scope (Hallucination Test)",
        "question": "According to the NIST AI Risk Management Framework, what is the mandatory cryptocurrency fine for deploying an unregistered AI model on Mars?",
        "ground_truth": "I don't know. The provided context does not contain information about cryptocurrency fines or deploying AI models on Mars.",
        "expected_keywords": ["don't know"]
    }
]

print(f"Loaded {len(test_dataset)} test cases across the 6 benchmark PDFs.")

Loaded 10 test cases across the 6 benchmark PDFs.


###Automated Evaluation###

In [20]:



# 1. Helper: Cosine Similarity (0.0 to 1.0) using your HuggingFace embedding model
def calculate_cosine_similarity(text1, text2):
    vec1 = np.array(embeddings.embed_query(text1))
    vec2 = np.array(embeddings.embed_query(text2))
    return float(np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2)))

# 2. Helper: Faithfulness Grader (0.0 = Hallucinated, 0.5 = Partial, 1.0 = Faithful)
def grade_faithfulness(context_str, question_str, answer_str):
    judge_prompt = f"""You are a strict grading judge. 
Compare the ANSWER against the CONTEXT below.
- Return ONLY the number 1.0 if the ANSWER is completely supported by the CONTEXT (or rightly says it doesn't know when context is irrelevant).
- Return ONLY the number 0.5 if the ANSWER is partially supported but adds minor unsupported details.
- Return ONLY the number 0.0 if the ANSWER makes up facts not in the CONTEXT.
Do not write any words, only output 0.0, 0.5, or 1.0.

CONTEXT: {context_str}
QUESTION: {question_str}
ANSWER: {answer_str}
SCORE:"""
    raw_score = llm.invoke(judge_prompt).strip()
    for val in ["1.0", "0.5", "0.0"]:
        if val in raw_score:
            return float(val)
    return 0.5

evaluation_results = []

print("Starting Naive RAG Evaluation...\n")

for i, item in enumerate(test_dataset, 1):
    q = item["question"]
    gt = item["ground_truth"]
    keywords = item["expected_keywords"]
    
    print(f"Running Test {i}/{len(test_dataset)}: {item['category']}...")
    
    # --- TIMER 1: Vector Retrieval Step ---
    t_ret_start = time.perf_counter()
    retrieved_docs = retriever.invoke(q)
    retrieval_time = time.perf_counter() - t_ret_start
    
    # --- TIMER 2: Context Formatting & Prompt Construction Step ---
    t_prompt_start = time.perf_counter()
    combined_context = format_docs_with_sources(retrieved_docs)
    formatted_prompt = prompt.invoke({"context": combined_context, "question": q})
    prompt_build_time = time.perf_counter() - t_prompt_start
    
    # --- TIMER 3: Pure LLM Generation Step (Llama 3) ---
    t_gen_start = time.perf_counter()
    raw_llm_response = llm.invoke(formatted_prompt)
    answer = StrOutputParser().invoke(raw_llm_response)
    llm_generation_time = time.perf_counter() - t_gen_start
    
    # Total End-to-End Pipeline Time
    total_pipeline_time = retrieval_time + prompt_build_time + llm_generation_time
    
    # --- ACCURACY METRICS ---
    # Metric A: Context Relevance (Average cosine similarity of retrieved chunks to question)
    chunk_similarities = [calculate_cosine_similarity(q, doc.page_content) for doc in retrieved_docs]
    context_relevance = float(np.mean(chunk_similarities)) if chunk_similarities else 0.0
    
    # Metric B: Context Keyword Recall (Percentage of expected keywords found)
    if item["category"] == "Out-of-Scope (Hallucination Test)":
        keyword_hits = [kw for kw in keywords if kw.lower() in answer.lower()]
    else:
        keyword_hits = [kw for kw in keywords if kw.lower() in combined_context.lower()]
    keyword_recall = len(keyword_hits) / len(keywords) if keywords else 1.0
    
    # Metric C: Answer Similarity to Ground Truth
    answer_similarity = calculate_cosine_similarity(gt, answer)
    
    # Metric D: Faithfulness (LLM Judge Score)
    faithfulness = grade_faithfulness(combined_context, q, answer)
    
    # Log the PDF filenames retrieved for inspection
    sources_used = ", ".join(
        sorted({os.path.basename(doc.metadata.get("source", "Unknown")) for doc in retrieved_docs})
    )
    
    evaluation_results.append({
        "Pipeline_Version": "Naive RAG (v1)",
        "Test_ID": i,
        "Category": item["category"],
        "Question": q,
        "Retrieval_Time_s": round(retrieval_time, 4),
        "Prompt_Build_Time_s": round(prompt_build_time, 4),
        "LLM_Generation_Time_s": round(llm_generation_time, 2),
        "Total_Pipeline_Time_s": round(total_pipeline_time, 2),
        "Context_Relevance": round(context_relevance, 3),
        "Keyword_Recall": round(keyword_recall, 2),
        "Answer_Similarity": round(answer_similarity, 3),
        "Faithfulness": faithfulness,
        "Sources_Retrieved": sources_used,
        "Generated_Answer": answer
    })
    
    print(f" -> Done in {total_pipeline_time:.2f}s (Retrieval: {retrieval_time:.3f}s | LLM: {llm_generation_time:.2f}s) | Recall: {keyword_recall:.2f} | Answer Sim: {answer_similarity:.3f}")

df_naive = pd.DataFrame(evaluation_results)
print("\nAll 10 Evaluation Tests Complete!")

Starting Naive RAG Evaluation...

Running Test 1/10: Specific Architecture (Original RAG Paper)...
 -> Done in 3.41s (Retrieval: 0.014s | LLM: 3.39s) | Recall: 0.00 | Answer Sim: 0.611
Running Test 2/10: Exact Technical Spec (Original RAG Paper)...
 -> Done in 3.49s (Retrieval: 0.014s | LLM: 3.48s) | Recall: 1.00 | Answer Sim: 0.813
Running Test 3/10: Broad Summary (Modular RAG Survey)...
 -> Done in 4.85s (Retrieval: 0.009s | LLM: 4.84s) | Recall: 0.75 | Answer Sim: 0.609
Running Test 4/10: Exact Numbers & Specs (Llama 3 Report)...
 -> Done in 5.20s (Retrieval: 0.011s | LLM: 5.19s) | Recall: 0.80 | Answer Sim: 0.843
Running Test 5/10: Architecture Detail (Transformer Paper)...
 -> Done in 3.55s (Retrieval: 0.011s | LLM: 3.53s) | Recall: 0.80 | Answer Sim: 0.909
Running Test 6/10: List Extraction (LLM Survey 140 Pages)...
 -> Done in 4.37s (Retrieval: 0.011s | LLM: 4.36s) | Recall: 0.00 | Answer Sim: 0.645
Running Test 7/10: Policy Guideline Lookup (NIST AI RMF)...
 -> Done in 4.46s (R

***Output of Tests***

In [21]:
# CELL 7: Generate Scoreboard & Export CSV Files for PowerPoint Graphing

# 1. Save detailed question-by-question results to CSV
DETAILED_CSV_PATH = "naive_rag_detailed_metrics.csv"
df_naive.to_csv(DETAILED_CSV_PATH, index=False)

# 2. Compute overall average scorecard across all 10 tests
numeric_cols = [
    "Retrieval_Time_s",
    "Prompt_Build_Time_s",
    "LLM_Generation_Time_s",
    "Total_Pipeline_Time_s",
    "Context_Relevance",
    "Keyword_Recall",
    "Answer_Similarity",
    "Faithfulness"
]

df_summary = pd.DataFrame([{
    "Pipeline_Version": "Naive RAG (v1)",
    **df_naive[numeric_cols].mean().round(4).to_dict()
}])

# 3. Save summary scorecard to CSV
SUMMARY_CSV_PATH = "naive_rag_summary_scorecard.csv"
df_summary.to_csv(SUMMARY_CSV_PATH, index=False)

print("==========================================================")
print("          NAIVE RAG (VERSION 1) FINAL SCORECARD           ")
print("==========================================================")
for col in numeric_cols:
    print(f"{col:<25}: {df_summary[col].iloc[0]}")
print("==========================================================")
print(f"Saved detailed logs to : {DETAILED_CSV_PATH}")
print(f"Saved summary averages : {SUMMARY_CSV_PATH}\n")

# 4. Display the clean scoreboard table in Jupyter
display_cols = [
    "Test_ID", 
    "Category", 
    "Retrieval_Time_s", 
    "LLM_Generation_Time_s", 
    "Total_Pipeline_Time_s", 
    "Context_Relevance", 
    "Keyword_Recall", 
    "Answer_Similarity", 
    "Faithfulness",
    "Sources_Retrieved"
]
df_naive[display_cols]

          NAIVE RAG (VERSION 1) FINAL SCORECARD           
Retrieval_Time_s         : 0.0119
Prompt_Build_Time_s      : 0.0004
LLM_Generation_Time_s    : 4.633
Total_Pipeline_Time_s    : 4.646
Context_Relevance        : 0.6622
Keyword_Recall           : 0.71
Answer_Similarity        : 0.7894
Faithfulness             : 0.6
Saved detailed logs to : naive_rag_detailed_metrics.csv
Saved summary averages : naive_rag_summary_scorecard.csv



,Test_ID,Category,Retrieval_Time_s,LLM_Generation_Time_s,Total_Pipeline_Time_s,Context_Relevance,Keyword_Recall,Answer_Similarity,Faithfulness,Sources_Retrieved
0,1,Specific Architecture (Original RAG Paper),0.0143,3.39,3.41,0.685,0.00,0.611,0.0,Modular_RAG_Comprehensive_Survey_2024.pdf
1,2,Exact Technical Spec (Original RAG Paper),0.0139,3.48,3.49,0.641,1.00,0.813,1.0,Original_RAG_Paper_Lewis_2020.pdf
2,3,Broad Summary (Modular RAG Survey),0.0094,4.84,4.85,0.593,0.75,0.609,0.5,"Modular_RAG_Comprehensive_Survey_2024.pdf, Ori..."
3,4,Exact Numbers & Specs (Llama 3 Report),0.0112,5.19,5.20,0.613,0.80,0.843,1.0,Llama_3_Herd_of_Models_Report_92_Pages.pdf
4,5,Architecture Detail (Transformer Paper),0.0110,3.53,3.55,0.669,0.80,0.909,1.0,Attention_Is_All_You_Need_Transformer.pdf
5,6,List Extraction (LLM Survey 140 Pages),0.0108,4.36,4.37,0.691,0.00,0.645,0.0,"LLM_Survey_140_Pages.pdf, Original_RAG_Paper_L..."
6,7,Policy Guideline Lookup (NIST AI RMF),0.0104,4.45,4.46,0.795,1.00,0.791,1.0,NIST_AI_Risk_Management_Framework_Policy.pdf
7,8,Multi-Item Policy List (NIST AI RMF),0.0098,4.54,4.55,0.786,1.00,0.905,1.0,NIST_AI_Risk_Management_Framework_Policy.pdf
8,9,Multi-Document Synthesis,0.0149,8.72,8.74,0.610,0.75,0.840,0.5,"Attention_Is_All_You_Need_Transformer.pdf, Lla..."
9,10,Out-of-Scope (Hallucination Test),0.0131,3.83,3.84,0.539,1.00,0.928,0.0,NIST_AI_Risk_Management_Framework_Policy.pdf
